In [ ]:

#@title ۱. نصب و راه‌اندازی  { display-mode: "form" }
import os
import base64
import time

print("⏳ در حال حذف نسخه‌های قدیمی و نصب نسخه بهینه GPU (1.26.0)...")

# نصب نسخه 1.26.0 که مخصوص CUDA 12 و کارت‌های کولب است
!pip uninstall -y -q onnxruntime onnxruntime-gpu insightface
!pip install -q onnxruntime-gpu==1.26.0
!pip install -q --no-deps insightface
!pip install -q opencv-python tqdm onnx

os.makedirs('/content/models', exist_ok=True)

def d(x):
    return base64.b64decode(x).decode('utf-8')

# دریافت مدل ارتقای چهره (GFPGAN 1.4)
_url = d("aHR0cHM6Ly9odWdnaW5nZmFjZS5jby9mYWNlZnVzaW9uL21vZGVscy0zLjAuMC9yZXNvbHZlL21haW4vZ2ZwZ2FuXzEuNC5vbm54")
_model_path = "/content/models/enhancer.onnx"

if not os.path.exists(_model_path) or os.path.getsize(_model_path) < 10000000:
    print("⏳ در حال دریافت مدل بهینه‌ساز چهره...")
    !curl -L -o {_model_path} "{_url}" --progress-bar
    if not os.path.exists(_model_path) or os.path.getsize(_model_path) < 10000000:
        !wget -q --show-progress -O {_model_path} "{_url}"

# لود پیش‌نیاز درایور گرافیک قبل از اجرای ONNX
import torch
import onnxruntime as ort

available_providers = ort.get_available_providers()
print(f"\n سخت‌افزارهای شناسایی‌شده: {available_providers}")

if 'CUDAExecutionProvider' in available_providers:
    print("✅ کارت گرافیک (CUDA/GPU) با موفقیت فعال شد و سرعت پردازش حداکثری خواهد بود.")
else:
    print("⚠️ هشدار: درایور شناسایی نشد.")

In [ ]:

#@title ۲. آپلود ویدیو مورد نظر { display-mode: "form" }
import os, shutil
from google.colab import files

if os.path.exists('/content/target.mp4'):
    os.remove('/content/target.mp4')

print("لطفاً فایل ویدیویی که می‌خواهید کیفیت آن ارتقا یابد را انتخاب کنید:")
uploaded = files.upload()
for f in uploaded.keys():
    shutil.move(f, '/content/target.mp4')
    print(f"✅ ویدیوی «{f}» با موفقیت بارگذاری شد.")
    break

In [ ]:

#@title ۳. شروع ارتقای کیفیت ویدیو { display-mode: "form" }
import cv2, os
import numpy as np
import onnxruntime as ort
import insightface
from insightface.app import FaceAnalysis
from tqdm import tqdm
from google.colab import files

target_video = '/content/target.mp4'
if not os.path.exists(target_video):
    raise FileNotFoundError("❌ فایل ویدیو پیدا نشد! لطفاً ابتدا سلول ۲ را اجرا کرده و ویدیو آپلود کنید.")

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
print("🚀 در حال بارگذاری ماژول‌ها روی کارت گرافیک...")

# ۱. بارگذاری ماژول تراز و تشخیص چهره
app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=0, det_size=(640, 640))

# ۲. راه‌اندازی سشن مدل GFPGAN با موتور ONNX روی GPU
enhancer_path = '/content/models/enhancer.onnx'
enhancer_session = ort.InferenceSession(enhancer_path, providers=providers)
enhancer_in_name = enhancer_session.get_inputs()[0].name

# ماتریس تراز ۵ نقطه‌ای چهره (۵۱۲×۵۱۲)
norm_512 = np.array([
    [192.93, 190.54],
    [318.55, 190.54],
    [256.00, 274.31],
    [201.26, 357.73],
    [310.74, 357.73]
], dtype=np.float32)

blend_mask = np.ones((512, 512), dtype=np.float32)
blend_mask = cv2.erode(blend_mask, np.ones((25, 25), np.uint8))
blend_mask = cv2.GaussianBlur(blend_mask, (45, 45), 0)
blend_mask = np.expand_dims(blend_mask, axis=-1)

def enhance_face(target_frame, kps):
    M, _ = cv2.estimateAffinePartial2D(kps, norm_512)
    if M is None: return target_frame
    IM = cv2.invertAffineTransform(M)
    warped = cv2.warpAffine(target_frame, M, (512, 512), borderMode=cv2.BORDER_REPLICATE)

    inp = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    inp = (inp - 0.5) / 0.5
    inp = np.transpose(inp, (2, 0, 1))[np.newaxis, ...]

    out = enhancer_session.run(None, {enhancer_in_name: inp})[0]
    out_img = np.clip((out[0].transpose(1, 2, 0) * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)
    out_img = cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR)

    enhanced_back = cv2.warpAffine(out_img, IM, (target_frame.shape[1], target_frame.shape[0]), borderMode=cv2.BORDER_REPLICATE)
    mask_back = cv2.warpAffine(blend_mask, IM, (target_frame.shape[1], target_frame.shape[0]))
    mask_back = np.expand_dims(mask_back, axis=-1)

    return (enhanced_back * mask_back + target_frame * (1.0 - mask_back)).astype(np.uint8)

# بررسی و باز کردن ویدیو
cap = cv2.VideoCapture(target_video)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

temp_output = '/content/temp_enhanced.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_output, fourcc, fps, (width, height))

print(f"🎬 شروع پردازش {total_frames} فریم با شتاب GPU...\n")
pbar = tqdm(total=total_frames, unit="frame")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # تشخیص چهره‌ها در فریم و اعمال ارتقا
    faces = app.get(frame)
    if faces:
        for f in faces:
            frame = enhance_face(frame, f.kps)

    out.write(frame)
    pbar.update(1)

cap.release()
out.release()
pbar.close()

# بازگرداندن صدای ویدیو با ffmpeg
final_output = '/content/enhanced_video.mp4'
print("\n🔊 در حال چسباندن مجدد صدا به ویدیو...")
!ffmpeg -y -i /content/temp_enhanced.mp4 -i /content/target.mp4 -c:v copy -c:a aac -map 0:v:0 -map 1:a:0? {final_output} -loglevel quiet 2>/dev/null || cp /content/temp_enhanced.mp4 {final_output}

print("🎉 تمام فریم‌ها با حداکثر کیفیت ارتقا یافتند!")
files.download(final_output)

برای ارتقا کیفیت عکس از سلول زیر استفاده کنید قبل از اجرا مطمئن شوید سلول اول (نصب) اجرا شده باشد

In [ ]:

#@title 📸 ارتقا کیفیت عکس
import cv2, os, shutil
import numpy as np
import torch
import onnxruntime as ort
import insightface
from insightface.app import FaceAnalysis
from google.colab import files
from IPython.display import display, Image

# بررسی وجود مدل از سلول ۱
enhancer_path = '/content/models/enhancer.onnx'
if not os.path.exists(enhancer_path):
    raise FileNotFoundError("❌ لطفاً ابتدا سلول ۱ را اجرا کنید تا مدل آماده باشد.")

print("لطفاً عکس مورد نظرتان (JPG یا PNG) را انتخاب و آپلود کنید:")
uploaded = files.upload()

if not uploaded:
    print("⚠️ هیچ فایلی انتخاب نشد.")
else:
    for filename in uploaded.keys():
        input_image_path = os.path.join('/content', filename)
        break

    # راه‌اندازی مدل‌ها روی کارت گرافیک
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    app = FaceAnalysis(name='buffalo_l', providers=providers)
    app.prepare(ctx_id=0, det_size=(640, 640))

    enhancer_session = ort.InferenceSession(enhancer_path, providers=providers)
    enhancer_in_name = enhancer_session.get_inputs()[0].name

    # ماتریس تراز ۵ نقطه‌ای چهره (۵۱۲×۵۱۲)
    norm_512 = np.array([
        [192.93, 190.54],
        [318.55, 190.54],
        [256.00, 274.31],
        [201.26, 357.73],
        [310.74, 357.73]
    ], dtype=np.float32)

    blend_mask = np.ones((512, 512), dtype=np.float32)
    blend_mask = cv2.erode(blend_mask, np.ones((25, 25), np.uint8))
    blend_mask = cv2.GaussianBlur(blend_mask, (45, 45), 0)
    blend_mask = np.expand_dims(blend_mask, axis=-1)

    def enhance_face(target_frame, kps):
        M, _ = cv2.estimateAffinePartial2D(kps, norm_512)
        if M is None: return target_frame
        IM = cv2.invertAffineTransform(M)
        warped = cv2.warpAffine(target_frame, M, (512, 512), borderMode=cv2.BORDER_REPLICATE)

        inp = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        inp = (inp - 0.5) / 0.5
        inp = np.transpose(inp, (2, 0, 1))[np.newaxis, ...]

        out = enhancer_session.run(None, {enhancer_in_name: inp})[0]
        out_img = np.clip((out[0].transpose(1, 2, 0) * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)
        out_img = cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR)

        enhanced_back = cv2.warpAffine(out_img, IM, (target_frame.shape[1], target_frame.shape[0]), borderMode=cv2.BORDER_REPLICATE)
        mask_back = cv2.warpAffine(blend_mask, IM, (target_frame.shape[1], target_frame.shape[0]))
        mask_back = np.expand_dims(mask_back, axis=-1)
        return (enhanced_back * mask_back + target_frame * (1.0 - mask_back)).astype(np.uint8)

    print("🚀 در حال بازسازی و شفاف‌سازی چهره با GPU...")
    img = cv2.imread(input_image_path)

    if img is None:
        print("❌ فایل تصویر خوانده نشد.")
    else:
        faces = app.get(img)
        if not faces:
            print("⚠️ چهره‌ای در تصویر شناسایی نشد.")
        else:
            # ارتقای تمام چهره‌های موجود در تصویر
            for f in faces:
                img = enhance_face(img, f.kps)

            output_image_path = '/content/enhanced_photo.png'
            cv2.imwrite(output_image_path, img)
            print(f"🎉 کیفیت {len(faces)} چهره ارتقا پیدا کرد!")

            # نمایش پیش‌نمایش تصویر نتیجه در خود کولب
            display(Image(output_image_path, width=450))

            # دانلود خودکار فایل باکیفیت
            files.download(output_image_path)